### 
PHASE: SQL ANALYSIS
#### Database: SQLite

In [1]:
import pandas as pd
import sqlite3
import re
from IPython.display import display

df = pd.read_csv(
    "E:\\OneDrive\\Desktop\\Istiyaque_Kamil\\Courses\\Data Analytics\\My Projects\\P1- E-Commerce Sales Analysis\\Data\\Ecommerce_sales_cleaned.csv"
)
# Coverting data types
df["order_date"]= pd.to_datetime(df["order_date"], format="mixed" )
df["ship_date"]= pd.to_datetime(df["ship_date"], format="mixed")



### STEP 1: PREPARE THE DATA FOR SQL


In [2]:
# Create a copy so the original DataFrame remains unchanged
sql_df=df.copy()

# Convert column names into SQL-friendly names
sql_df.columns = [
    re.sub(r"_+", "_",
          re.sub(r"[^a-z0-9]+", "_",col.strip().lower())
    ).strip("_")
    for col in sql_df.columns
]

# Create a temporary SQLite database in memory
conn = sqlite3.connect(":memory:")

# Transfer the DataFrame into a SQL table
sql_df.to_sql("sales", conn, if_exists="replace", index=False)

print("SQL Data Base Created Successfully")
print("Table name: sales")
print(f"Record loaded: {len(sql_df):,}")

SQL Data Base Created Successfully
Table name: sales
Record loaded: 10,194


### STEP 2: CREATE A REUSABLE SQL RUNNER


In [3]:
def run_query(title, query):
    print(title)

    result= pd.read_sql_query(query, conn)
    display(result)
    return result
    

### Q1. WHICH PRODUCTS HAVE HIGH SALES BUT LOW PROFITABILITY?
#### Concepts: SELECT, GROUP BY, HAVING, ORDER BY, CTE

In [4]:
q1= """
WITH  product_performance AS(
    SELECT
        product_id,
        product_name,
        SUM(sales) AS total_sales,
        SUM(profit) AS total_profit,
        SUM(quantity) AS total_quantity
    FROM sales
    GROUP BY product_id, product_name
),
overall AS(
    SELECT
        SUM(sales) * 1.0 / SUM(profit) AS sales_profit_ratio
    FROM sales
)

SELECT
    product_id,
    product_name,
    ROUND(total_sales, 2) AS total_sales,
    ROUND(total_profit, 2) AS total_profit,
    ROUND(
        total_profit * 100.0 / NULLIF(total_Sales, 0), 2
        ) AS profit_margin_pct,
        total_quantity
        
FROM product_performance
WHERE total_sales > (
    SELECT AVG (total_sales)
    FROM product_performance
)

AND total_profit * 1.0 / NULLIF(total_Sales, 0) <
    (SELECT SUM(profit)*1.0 / SUM(sales) FROM sales)
ORDER BY total_sales DESC
LIMIT 10;
"""

high_sales_low_profit = run_query(
    "Q1 — High-Sales Products with Weak Profit Margins",
    q1
)


Q1 — High-Sales Products with Weak Profit Margins


,product_id,product_name,total_sales,total_profit,profit_margin_pct,total_quantity
0,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,-8.00,6
1,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,21870.58,0.00,0.00,39
2,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,19823.48,2233.51,11.27,37
3,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98,4.00,48
4,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,-10.45,27
5,OFF-SU-10000151,High Speed Automatic Electric Letter Opener,17030.31,-262.00,-1.54,11
6,TEC-MA-10000822,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97,-27.27,18
7,OFF-SU-10002881,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18,-7.80,22
8,FUR-BO-10004834,"Riverside Palais Royal Lawyers Bookcase, Royal...",15610.97,-669.54,-4.29,24
9,FUR-TA-10003473,Bretford Rectangular Conference Table Tops,13565.57,-359.35,-2.65,53


### Q2. WHICH INDIVIDUAL PRODUCTS GENERATE THE MOST PROFIT?
#### Concepts: GROUP BY, SUM, ORDER BY, LIMIT

In [5]:
q2 = """
SELECT
    product_id,
    product_name,
    ROUND(SUM(sales),2) AS total_sales,
    ROUND(SUM(profit),2) AS total_profit,
    ROUND(
        SUM(profit)*100.0 / NULLIF(SUM(sales),0),2
    ) AS profit_margin_pct
FROM sales
GROUP BY product_id, product_name
ORDER BY total_profit DESC
LIMIT 10;
"""

top_product = run_query(
    "Q2 Top 10 Product by Total Profit",
    q2
)

Q2 Top 10 Product by Total Profit


,product_id,product_name,total_sales,total_profit,profit_margin_pct
0,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93,40.91
1,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04,28.24
2,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88,37.07
3,TEC-CO-10003763,Canon PC1060 Personal Laser Copier,11619.83,4570.93,39.34
4,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,22.29
5,TEC-MA-10003979,Ativa V4110MDD Micro-Cut Shredder,7699.89,3772.95,49.00
6,TEC-MA-10001047,"3D Systems Cube Printer, 2nd Generation, Magenta",14299.89,3717.97,26.00
7,TEC-AC-10002049,Plantronics Savi W720 Multi-Device Wireless He...,9367.29,3696.28,39.46
8,OFF-BI-10001120,Ibico EPK-21 Electric Binding System,15875.92,3345.28,21.07
9,TEC-MA-10000045,Zebra ZM400 Thermal Label Printer,6965.70,3343.54,48.00


### Q3. WHICH PRODUCTS GENERATE THE LOWEST PROFIT?
#### Concepts: WHERE, GROUP BY, ORDER BY

In [6]:
q3="""
SELECT
    product_id,
    product_name,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(
        SUM(profit) * 100.0 / NULLIF(SUM(sales), 0),2
    ) AS profit_margin_pct

FROM  sales
GROUP BY product_id, product_name
ORDER  BY profit_margin_pct ASC
LIMIT 10

"""

bottom_products = run_query("Q3 — 10 Lowest-Profit Products",
    q3
)



Q3 — 10 Lowest-Profit Products


,product_id,product_name,total_sales,total_profit,profit_margin_pct
0,OFF-AP-10002203,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.62,-4.47,-275.00
1,FUR-BO-10001567,"Bush Westfield Collection Bookcases, Dark Cher...",90.88,-190.85,-210.00
2,OFF-AP-10000390,Euro Pro Shark Stick Mini Vacuum,170.74,-325.63,-190.71
3,TEC-MA-10003337,Okidata B401 Printer,179.99,-251.99,-140.00
4,TEC-MA-10001695,Zebra GK420t Direct Thermal/Thermal Transfer P...,703.71,-938.28,-133.33
5,OFF-BI-10003166,GBC Plasticlear Binding Covers,68.88,-68.42,-99.33
6,TEC-MA-10000904,"Brother MFC-9340CDW LED All-In-One Printer, Co...",341.99,-319.19,-93.33
7,TEC-MA-10002210,Epson TM-T88V Direct Thermal Printer - Monochr...,1212.70,-1057.23,-87.18
8,TEC-MA-10004521,Epson Perfection V600 Photo Scanner,206.99,-172.49,-83.33
9,OFF-BI-10000778,GBC VeloBinder Electric Binding Machine,496.02,-411.33,-82.93


### Q4. HOW DO SHIPPING MODES COMPARE?
#### Concepts: GROUP BY, AVG, date calculations

In [7]:
q4="""

SELECT
    ship_mode,
    COUNT(DISTINCT order_id) AS unique_orders,
    ROUND(SUM(sales),2) AS total_sales,
    ROUND(SUM(profit),2) AS total_profit,
    ROUND(
        SUM(profit *100.0)/ NULLIF(SUM(sales),0),2
        ) AS profit_margin_pct,

    ROUND(
        AVG(julianday(ship_date)- julianday(order_date)), 2
    ) AS average_shipping_days

FROM sales
WHERE order_date IS NOT NULL
    AND ship_date IS NOT NULL
GROUP BY ship_mode
ORDER BY average_shipping_days ASC;
"""

shipping_analysis = run_query("Q4 — Shipping Mode Performance",
    q4
)


Q4 — Shipping Mode Performance


,ship_mode,unique_orders,total_sales,total_profit,profit_margin_pct,average_shipping_days
0,Same Day,266,129271.96,16160.75,12.50,0.04
1,First Class,795,351750.74,49012.72,13.93,2.18
2,Second Class,982,466671.11,58962.13,12.63,3.24
3,Standard Class,3068,1378840.55,168161.21,12.20,4.99


### Q5. WHICH CUSTOMERS GENERATE THE MOST VALUE?
#### Concepts: GROUP BY, COUNT DISTINCT, HAVING, ORDER BY



In [8]:
q5 = """
SELECT
    customer_id,
    customer_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(
        SUM(profit) *100.0 / NULLIF (SUM(sales), 0),2
    ) AS profit_margin_pct

FROM sales
GROUP BY customer_id, customer_name
HAVING COUNT (DISTINCT order_id) >= 2
ORDER BY total_sales DESC
LIMIT 10;
"""

top_customers = run_query(
    "Q5 - Top 10 Repeat Customers by Sales",
    q5
    )

Q5 - Top 10 Repeat Customers by Sales


,customer_id,customer_name,total_orders,total_sales,total_profit,profit_margin_pct
0,SM-20320,Sean Miller,5,25043.05,-1980.74,-7.91
1,TC-20980,Tamara Chand,5,19052.22,8981.32,47.14
2,RB-19360,Raymond Buch,6,15117.34,6976.10,46.15
3,TA-21385,Tom Ashbrook,4,14595.62,4703.79,32.23
4,AB-10105,Adrian Barton,10,14473.57,5444.81,37.62
5,KL-16645,Ken Lonsdale,12,14175.23,806.85,5.69
6,SC-20095,Sanjit Chand,9,14142.33,5757.41,40.71
7,HL-15040,Hunter Lopez,6,12873.30,5622.43,43.68
8,SE-20110,Sanjit Engle,11,12209.44,2650.68,21.71
9,CC-12370,Christopher Conant,5,12129.07,2177.05,17.95


### Q6. WHICH STATES HAVE HIGH SALES BUT WEAK PROFITABILITY?
#### Concepts: GROUP BY, HAVING, calculated columns

In [9]:
q6 = """
SELECT
    state_province,
    COUNT (DISTINCT order_id) AS unique_orders,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) *100.0 / NULLIF(SUM(sales),0),2
    ) AS profit_margin_pct
FROM sales
GROUP BY state_province
HAVING SUM(sales) > (
    SELECT AVG(state_sales)
    FROM(
        SELECT SUM(sales) AS state_sales
        FROM sales
        GROUP BY state_province
    )
)
    
ORDER BY profit_margin_pct ASC
LIMIT 10;
"""

state_analysis = run_query(
    "Q6 — High-Sales States with the Weakest Profit Margins",
    q6
)



Q6 — High-Sales States with the Weakest Profit Margins


,state_province,unique_orders,total_sales,total_profit,profit_margin_pct
0,Ohio,236,78258.14,-16971.38,-21.69
1,Illinois,276,80166.10,-12607.89,-15.73
2,Texas,487,170188.05,-25729.36,-15.12
3,North Carolina,136,55603.16,-7490.91,-13.47
4,Pennsylvania,288,116511.91,-15559.96,-13.35
5,Florida,200,89473.71,-3399.30,-3.80
6,California,1021,457687.63,76381.39,16.69
7,New York,562,310876.27,74038.55,23.82
8,Washington,256,138641.27,33402.65,24.09
9,Virginia,115,70636.72,18597.95,26.33


### Q7. HOW ARE ORDERS DISTRIBUTED BY ORDER VALUE?
#### Concepts: CTE, CASE WHEN, GROUP BY

In [10]:
q7 = """
WITH order_values AS (
    SELECT
        order_id,
        SUM(sales) AS order_value,
        SUM(profit) AS order_profit
    FROM sales
    GROUP BY order_id
),
order_bands AS (
    SELECT
        order_id,
        order_value,
        order_profit,
        CASE
            WHEN order_value < 100 THEN 'Below 100'
            WHEN order_value < 500 THEN '100 to 499'
            WHEN order_value < 1000 THEN '500 to 999'
            ELSE '1000 and Above'
        END AS order_value_band
    FROM order_values
)

SELECT
    order_value_band,
    COUNT(*) AS number_of_orders,
    ROUND(AVG(order_value), 2) AS average_order_value,
    ROUND(SUM(order_value), 2) AS total_sales,
    ROUND(SUM(order_profit), 2) AS total_profit
FROM order_bands
GROUP BY order_value_band
ORDER BY
    CASE order_value_band
        WHEN 'Below 100' THEN 1
        WHEN '100 to 499' THEN 2
        WHEN '500 to 999' THEN 3
        ELSE 4
    END;
"""

order_value_analysis = run_query(
    "Q7 - Order Value Distribution",
    q7
)

Q7 - Order Value Distribution


,order_value_band,number_of_orders,average_order_value,total_sales,total_profit
0,Below 100,2173,36.58,79480.24,11861.35
1,100 to 499,1644,253.07,416041.20,44697.03
2,500 to 999,666,706.07,470241.16,41791.27
3,1000 and Above,628,2166.83,1360771.75,193947.17


### Q8. WHICH ORDERS GENERATED THE LARGEST LOSSES?
#### Concepts: HAVING, aggregate calculations, ORDER BY

In [11]:
q8= """
SELECT
    order_id,
    COUNT(*) AS product_records,
    ROUND (SUM(sales), 2) AS order_sales,
    ROUND (SUM(profit), 2) AS order_profit
FROM sales
GROUP BY order_id
HAVING SUM(profit) < 0
ORDER BY order_profit ASC
LIMIT 10;
"""

loss_making_orders = run_query (
    "Q8 - 10 Orders with the Largest Total Losses",
    q8
)


Q8 - 10 Orders with the Largest Total Losses


,order_id,product_records,order_sales,order_profit
0,US-2025-108196,3,5016.55,-6892.37
1,US-2026-168116,2,8167.42,-3825.34
2,US-2023-169019,6,2656.72,-3791.16
3,US-2026-134845,5,2613.31,-3424.35
4,US-2026-122714,1,1889.99,-2929.48
5,US-2026-131254,3,1729.29,-2330.27
6,US-2024-147830,4,4190.21,-1980.38
7,US-2023-139892,7,10539.90,-1878.79
8,US-2024-116638,1,4297.64,-1862.31
9,US-2025-130946,3,1616.70,-1790.27


### STEP 3: CLOSE THE DATABASE WHEN FINISHED

In [12]:
print("\nSQL ANALYSIS COMPLETE")
print("Queries executed: 8")
print("Review the result tables above before documenting findings.")

conn.close()


SQL ANALYSIS COMPLETE
Queries executed: 8
Review the result tables above before documenting findings.
